# Tutorial del árbol genealógico con UnifyWeaver

Este cuaderno de trabajo interactivo demuestra cómo usar UnifyWeaver para compilar predicados de Prolog a scripts de Bash.

## Requisitos previos

- SWI-Prolog instalado
- Biblioteca UnifyWeaver disponible
- Kernel de Jupyter para Prolog instalado (`pip install prolog-jupyter-kernel`)

## Objetivos de aprendizaje

Al final de este cuaderno de trabajo, podrás:
1. Definir hechos y reglas de Prolog
2. Usar UnifyWeaver para compilar predicados a Bash
3. Probar los scripts de Bash generados
4. Comprender la compilación de clausuras transitivas

## Paso 1: Inicializar el entorno de UnifyWeaver

En primer lugar, necesitamos cargar los módulos de UnifyWeaver. Usaremos el archivo `init.pl` del directorio education.

In [ ]:
% Load the initialization file
['../init'].

## Paso 2: Definir relaciones familiares

Definamos algunas relaciones progenitor-hijo del árbol genealógico bíblico.

In [ ]:
% Define parent facts
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Paso 3: Probar consultas de parent

Antes de compilar, verifiquemos que nuestros datos sean correctos con algunas consultas de Prolog.

In [ ]:
% Query: Who are Abraham's children?
parent(abraham, Child).

In [ ]:
% Query: Who are Jacob's children?
parent(jacob, Child).

## Paso 4: Definir la relación ancestor

Ahora definamos la clausura transitiva: la relación `ancestor`.

In [ ]:
% Define ancestor as transitive closure of parent
:- dynamic ancestor/2.

% Base case: parent is an ancestor
ancestor(X, Y) :- parent(X, Y).

% Recursive case: if X is parent of Y and Y is ancestor of Z, then X is ancestor of Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Paso 5: Probar consultas de ancestor

Verifiquemos que nuestro predicado ancestor funcione correctamente.

In [ ]:
% Query: Is Abraham an ancestor of Jacob?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Query: Who are all of Abraham's descendants?
ancestor(abraham, Descendant).

## Paso 6: Compilar parent a Bash

Ahora la parte emocionante: ¡compilemos nuestros hechos `parent/2` a un script de Bash!

In [ ]:
% Load the stream compiler
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compile parent facts to bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Paso 7: Guardar el script de parent

Guardemos el código Bash generado en un archivo.

In [ ]:
% Save to file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Paso 8: Compilar ancestor a Bash

Ahora compilemos el predicado `ancestor/2`, que utiliza recursión.

In [ ]:
% Load the recursive compiler
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compile ancestor to bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Paso 9: Guardar el script de ancestor

Guarda el script de ancestor en un archivo.

In [ ]:
% Save to file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Paso 10: Probar los scripts generados

¡Ahora probemos nuestros scripts de Bash generados! Usaremos la magia `%%bash` para ejecutar comandos de bash.

In [ ]:
%%bash
# Source the parent script
source ../output/parent.sh

# Test: Who are Abraham's children?
echo "Abraham's children:"
parent abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Who are Abraham's descendants?
echo "Abraham's descendants:"
ancestor abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Is Abraham an ancestor of Judah?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Yes, Abraham is an ancestor of Judah"
else
    echo "✗ No"
fi

## Paso 11: Comprender la estrategia de compilación

Analicemos lo que hizo UnifyWeaver:

1. **Compilación de parent**: Utilizó `stream_compiler` para crear una función de transmisión simple que emite todos los pares progenitor-hijo

2. **Compilación de ancestor**: Detectó el patrón de clausura transitiva y utilizó la optimización BFS (recorrido de grafos en anchura) para computar eficientemente todos los ancestros alcanzables

Verifiquemos la estrategia de compilación:

In [ ]:
% Check if ancestor is classified as recursive
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Resumen

En este cuaderno de trabajo, aprendiste:

✅ Cómo definir hechos y reglas de Prolog

✅ Cómo usar `stream_compiler` de UnifyWeaver para hechos

✅ Cómo usar `recursive_compiler` de UnifyWeaver para predicados recursivos

✅ Cómo probar scripts de Bash generados

✅ Que UnifyWeaver detecta automáticamente la clausura transitiva y aplica la optimización BFS

## Próximos pasos

Prueba estos ejercicios:

1. Agregar más miembros familiares al árbol
2. Definir un predicado `grandparent/2` y compilarlo
3. Crear un predicado `sibling/2` (dos personas con el mismo progenitor)
4. Explorar el código Bash generado para comprender el algoritmo BFS

¡Continúa con el **Cuaderno de trabajo 2: Comparación de patrones de recursión** para aprender sobre patrones avanzados de recursión!